In [5]:
from datasets import load_dataset

dataset = load_dataset("json", data_files={
    "train": "train_kichwa_es.jsonl",
    "validation": "valid_kichwa_es.jsonl",
    "test": "test_kichwa_es.jsonl"
})

print(dataset)
print(dataset["train"][0])

Generating train split: 2300 examples [00:00, 124653.05 examples/s]
Generating validation split: 287 examples [00:00, 17461.31 examples/s]
Generating test split: 288 examples [00:00, 47910.19 examples/s]

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 2300
    })
    validation: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 287
    })
    test: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 288
    })
})
{'instruction': 'Traduce del Kichwa al Español', 'input': 'Ñawpa pachapa shamuk pachapa hampikkuna', 'output': 'Píldoras para antes y para después.'}


In [3]:
print(dataset["train"][5])

{'instruction': 'Traduce del Kichwa al Español', 'input': 'Ari, niway, Fredy, yalliy, yalliy.', 'output': 'Dime, Fredy, pasa, pasa.'}


In [4]:
def format_data(example):
    return {
        "input_text": example["input"],
        "target_text": example["output"]
    }

dataset = dataset.map(format_data)

In [5]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

In [6]:
def preprocess(example):
    model_inputs = tokenizer(
        example["input_text"],
        max_length=128,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        example["target_text"],
        max_length=128,
        truncation=True,
        padding="max_length"
    )

    labels_ids = labels["input_ids"]
    labels_ids = [l if l != tokenizer.pad_token_id else -100 for l in labels_ids]

    model_inputs["labels"] = labels_ids
    return model_inputs

tokenized_dataset = dataset.map(preprocess)

Map: 100%|██████████| 288/288 [00:00<00:00, 2373.31 examples/s]


In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./results_flan_v2",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=10,
    learning_rate=2e-5,
    logging_steps=10,
    save_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"]
)

trainer.train()

  0%|          | 11/8625 [00:04<44:55,  3.20it/s] 

{'loss': 3.5953, 'grad_norm': 5.863841533660889, 'learning_rate': 1.99768115942029e-05, 'epoch': 0.02}


  0%|          | 21/8625 [00:07<42:51,  3.35it/s]

{'loss': 3.4522, 'grad_norm': 7.999049186706543, 'learning_rate': 1.99536231884058e-05, 'epoch': 0.03}


  0%|          | 31/8625 [00:10<42:30,  3.37it/s]

...

 53%|█████▎    | 4600/8625 [27:44<39:33,  1.70it/s]

{'loss': 1.8852, 'grad_norm': 4.920433521270752, 'learning_rate': 9.333333333333334e-06, 'epoch': 8.0}


                                                   
 53%|█████▎    | 4600/8625 [27:51<39:33,  1.70it/s]

{'eval_loss': 1.8473844528198242, 'eval_runtime': 6.9463, 'eval_samples_per_second': 41.317, 'eval_steps_per_second': 10.365, 'epoch': 8.0}


RuntimeError: [enforce fail at inline_container.cc:603] . unexpected pos 74816 vs 74708

In [6]:
trainer.save_model("./modelo_kichwa_flan_t5")
tokenizer.save_pretrained("./modelo_kichwa_flan_t5")

print("Modelo guardado correctamente.")

NameError: name 'trainer' is not defined

In [ ]:
def traducir(texto, model, tokenizer, max_len=80):
    inputs = tokenizer(texto, return_tensors="pt", truncation=True)


    inputs = inputs.to(model.device)

    outputs = model.generate(
        **inputs,
        max_length=max_len,
        num_beams=5,
        early_stopping=True,
        no_repeat_ngram_size=2
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [12]:
predicciones = []
referencias = []
entradas = []

for example in dataset["test"]:
    entrada = example["input"]
    referencia = example["output"]
    prediccion = traducir(entrada, model, tokenizer)

    entradas.append(entrada)
    referencias.append(referencia)
    predicciones.append(prediccion)

print("Predicciones generadas correctamente.")

Predicciones generadas correctamente.


In [13]:
import pandas as pd

df_resultados = pd.DataFrame({
    "input_kichwa": entradas,
    "referencia_es": referencias,
    "prediccion_es": predicciones
})

df_resultados.to_csv("predicciones_flan.csv", index=False, encoding="utf-8-sig")
print("Archivo guardado correctamente.")

Archivo guardado correctamente.


In [15]:
import sacrebleu

bleu = sacrebleu.corpus_bleu(predicciones, [referencias])
print("BLEU:", bleu.score)

BLEU: 3.2652422708392392


## DEMO

In [9]:
%pip install gtts


     ---------------------------------------- 98.2/98.2 kB 1.9 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.4.1
    Uninstalling click-8.4.1:
      Successfully uninstalled click-8.4.1
Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
huggingface-hub 1.17.0 requires click>=8.4.0, but you have click 8.1.8 which is incompatible.
typer 0.25.1 requires click>=8.2.1, but you have click 8.1.8 which is incompatible.

[notice] A new release of pip available: 22.3.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [39]:
# =========================
# 1. IMPORTS
# =========================
import os
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from gtts import gTTS

# =========================
# 2. CONFIGURACIÓN
# =========================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Ruta de tu modelo ENTRENADO
MODEL_PATH = "./modelo_kichwa_flan_t5"   # <-- CAMBIA si es necesario

# =========================
# 3. CARGAR MODELO
# =========================
print("Cargando modelo...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATH).to(DEVICE)

print("Modelo cargado correctamente")

Cargando modelo...


Loading weights: 100%|██████████| 282/282 [00:00<00:00, 11274.47it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Modelo cargado correctamente


In [40]:
def traducir_kichwa(texto):
    prompt = (
        f"Instruction: Traduce del Kichwa al Español\n"
        f"Input: {texto}\n"
        f"Output:"
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=128
    ).to(DEVICE)

    outputs = model.generate(
        **inputs,
        max_length=128,
        num_beams=4
    )

    return tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

In [41]:
ejemplos = [
    "Alli chishi, pushak alcaldesa.",
    "Sacha mama shina wayllata.",
    "Ingléstapash yachachichun, alcalde.",
    "Imanallatak kanki, shamupay, yaykuripay, tiyaripay."
]

for e in ejemplos:
    print("="*50)
    print("INPUT :", e)
    print("OUTPUT:", traducir_kichwa(e))

INPUT : Alli chishi, pushak alcaldesa.
OUTPUT: Gracias, alcaldesa.
INPUT : Sacha mama shina wayllata.
OUTPUT: Y esa mamá es una mujer.
INPUT : Ingléstapash yachachichun, alcalde.
OUTPUT: En inglés, alcalde.
INPUT : Imanallatak kanki, shamupay, yaykuripay, tiyaripay.
OUTPUT: Pero eso no es cierto, nios, nios, nios.


In [11]:
# =========================
# 4. FUNCIÓN DE TRADUCCIÓN
# =========================
def traducir(texto):
    entrada = "Traduce del Kichwa al Español: " + texto

    inputs = tokenizer(
        entrada,
        return_tensors="pt",
        truncation=True,
        padding=True
    ).to(DEVICE)

    outputs = model.generate(
        **inputs,
        max_length=80,
        num_beams=5,
        early_stopping=True,
        no_repeat_ngram_size=2
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)


In [12]:
#=========================
# 5. FUNCIÓN TEXTO → AUDIO
# =========================
def texto_a_audio(texto, nombre_archivo="salida.mp3"):
    tts = gTTS(text=texto, lang="es")
    tts.save(nombre_archivo)

    # Reproducir automáticamente (Windows)
    os.system(f"start {nombre_archivo}")

In [13]:
# =========================
# 6. FUNCIÓN DEMO COMPLETA
# =========================
def demo(audio_path, texto_kichwa):
    print("\n==============================")
    print("🎧 Reproduciendo audio...")
    os.system(f"start {audio_path}")

    print("\n📝 Texto en Kichwa:")
    print(texto_kichwa)

    print("\n🔄 Traduciendo...")
    traduccion = traducir(texto_kichwa)

    print("\n🇪🇸 Traducción:")
    print(traduccion)

    print("\n🔊 Generando audio en español...")
    texto_a_audio(traduccion)


In [14]:

# =========================
# 7. EJEMPLOS (USA TUS AUDIOS)
# =========================
ejemplos = [
    {
        "audio": "2.mp3",
        "texto": "pandemiamanta rimaykuna"
    },
    {
        "audio": "3.mp3",
        "texto": "alli pacha"
    }
]

In [15]:
# =========================
# 8. EJECUTAR DEMO
# =========================
for ej in ejemplos:
    demo(ej["audio"], ej["texto"])

    input("\nPresiona ENTER para continuar...")


🎧 Reproduciendo audio...

📝 Texto en Kichwa:
pandemiamanta rimaykuna

🔄 Traduciendo...

🇪🇸 Traducción:
Traduce del Kichwa al Espaol: la pandemia

🔊 Generando audio en español...

🎧 Reproduciendo audio...

📝 Texto en Kichwa:
alli pacha

🔄 Traduciendo...

🇪🇸 Traducción:
Traduce del Kichwa al Espaol

🔊 Generando audio en español...


## flan t5 base sin fine tuning

In [27]:
%pip install transformers datasets sacrebleu evaluate sentencepiece torch

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [28]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(DEVICE)

print("Modelo cargado correctamente")

c:\Users\marck\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\marck\.cache\huggingface\hub\models--google--flan-t5-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 282/282 [00:00<00:00, 9108.22it/s]
[transformers] The tied w

Modelo cargado correctamente


In [29]:
import json

test_data = []

with open("test_kichwa_es.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        test_data.append(json.loads(line))

print("Ejemplos:", len(test_data))

Ejemplos: 288


In [30]:
def translate_kichwa(text):

    prompt = f"Traduce del Kichwa al Español: {text}"

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=128
    ).to(DEVICE)

    outputs = model.generate(
        **inputs,
        max_length=128,
        num_beams=4,
        early_stopping=True
    )

    prediction = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return prediction

In [31]:
predictions = []
references = []

for sample in test_data:

    pred = translate_kichwa(sample["input"])

    predictions.append(pred)
    references.append(sample["output"])

print("Predicciones generadas:", len(predictions))

Predicciones generadas: 288


In [32]:
for i in range(10):

    print("="*60)
    print("Kichwa:")
    print(test_data[i]["input"])

    print("\nReferencia:")
    print(references[i])

    print("\nPredicción:")
    print(predictions[i])

Kichwa:
Imanallatak kanki, shamupay, yaykuripay, tiyaripay.

Referencia:
¿Cómo estás?, venga, entre, siéntese.

Predicción:
Traduce del Kichwa al Espaol: Imanallatak kanki, shamupay, yaykuripay, tiyaripay.
Kichwa:
Hampik, allichu kakrini?

Referencia:
¿Doctor, voy a estar bien?

Predicción:
Traduce del Kichwa al Espaol
Kichwa:
Shinallatak llakta mikuyta mikunchik.

Referencia:
Se come del campo.

Predicción:
Traduce del Kichwa al Espaol: Shinallatak llakta mikuyta mikunchik.
Kichwa:
Rikuychik, rikuychik, chayta chapamukukka Apoloniomi kan, payka raku runa, antawakunawan llamkan.

Referencia:
Vean, vean, aquel que se asoma es el gordo Apolonio, el mecánico.

Predicción:
Traduce del Kichwa al Espaol: Rikuychik, rikuychik, chayta chapamukukka Apoloniomi kan, payka raku runa, antawakunawan llamkan.
Kichwa:
Kipamanmi llakikuna chayamun.

Referencia:
Luego viene el desastre.

Predicción:
Traduce del Kichwa al Espaol: Kipamanmi llakikuna chayamun.
Kichwa:
Shuyay, mashi.

Referencia:
Espere, s

In [35]:
text = "Mana entendenichu"

prompt = f"Traduce del Kichwa al Español: {text}"

inputs = tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True
).to(DEVICE)

outputs = model.generate(
    **inputs,
    max_new_tokens=50,
    num_beams=4
)

prediction = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("Entrada:", text)
print("Predicción:", prediction)

Entrada: Mana entendenichu
Predicción: Traduce del Kichwa al Espaol: Mana entendenichu


In [33]:
import evaluate

bleu = evaluate.load("bleu")

bleu_result = bleu.compute(
    predictions=predictions,
    references=[[r] for r in references]
)

print("\nBLEU:")
print(bleu_result["bleu"])


BLEU:
0.013615681471908178


In [34]:
chrf = evaluate.load("chrf")

chrf_result = chrf.compute(
    predictions=predictions,
    references=references,
    word_order=2
)

print("\nchrF++:")
print(chrf_result["score"])


chrF++:
16.658915623102946


In [42]:
from evaluate import load
bleu = load("bleu")
chrf = load("chrf")
